# Multi-step sales forecasting

An end-to-end example predicting multiple future time steps of a sales dataset with `tfts`.

In [2]:
import logging
import os
from typing import List, Optional, Union

import numpy as np
import pandas as pd
import tensorflow as tf

from tfts import AutoConfig, AutoModel, AutoModelForForecasting, KerasTrainer

In [3]:
class CFG:
    input_dir = "/kaggle/input/china-vehicle-sales-data/china_vehicle_sales_data.csv"
    train_sequence_length = 12
    predict_sequence_length = 3

In [4]:
if os.path.exists(CFG.input_dir):
    data = pd.read_csv(CFG.input_dir)
else:
    # Keep the notebook runnable outside Kaggle with a small deterministic
    # vehicle-sales-shaped dataset. Replace this with the Kaggle CSV when
    # reproducing the original experiment.
    dates = pd.date_range("2020-01-01", periods=48, freq="MS")
    rows = []
    for province in range(2):
        for model_name in ("compact", "suv"):
            base = 100 + 20 * province + (30 if model_name == "suv" else 0)
            rows.extend(
                {"provinceId": province, "model": model_name, "Date": date, "salesVolume": base + 10 * np.sin(i / 3)}
                for i, date in enumerate(dates)
            )
    data = pd.DataFrame(rows)

data

,provinceId,model,Date,salesVolume
0,0,compact,2020-01-01,100.000000
1,0,compact,2020-02-01,103.271947
2,0,compact,2020-03-01,106.183698
3,0,compact,2020-04-01,108.414710
4,0,compact,2020-05-01,109.719379
...,...,...,...,...
187,1,suv,2023-08-01,159.808210
188,1,suv,2023-09-01,158.630599
189,1,suv,2023-10-01,156.502878
190,1,suv,2023-11-01,153.659282


In [5]:
# https://github.com/hongyingyue/Vehicle-sales-predictor/blob/main/vehicle_ml/feature/ts_feature.py

logger = logging.getLogger(__name__)


def add_lagging_feature(
    data: pd.DataFrame,
    groupby_column: Union[str, List[str]],
    value_columns: List[str],
    lags: List[int],
    feature_columns: Optional[List[str]] = None,
):
    # note that the data should be sorted by time already
    # the lagging feature could be further developed use f1 - f1_lag, or f1 / f1_lag

    if not isinstance(groupby_column, (str, list)):
        raise TypeError(f"'groupby_column' must be a string or a list of strings, but got {type(groupby_column)}.")

    if not isinstance(value_columns, (list, tuple)):
        raise TypeError(f"'value_columns' must be a list of strings, but got {type(value_columns)}.")

    feature_columns: List[str] = feature_columns if feature_columns is not None else []
    for column in value_columns:
        if column not in data.columns:
            raise ValueError(f"Value column '{column}' not found in DataFrame.")

        for lag in lags:
            feature_col_name = f"{column}_lag{lag}"
            feature_columns.append(feature_col_name)
            data[feature_col_name] = data.groupby(groupby_column)[column].shift(lag)
    return data

In [6]:
feature_columns = []

data = add_lagging_feature(
    data,
    groupby_column=["provinceId", "model"],
    value_columns=["salesVolume"],
    lags=list(range(1, 12)),
    feature_columns=feature_columns,
)

data

,provinceId,model,Date,salesVolume,salesVolume_lag1,salesVolume_lag2,salesVolume_lag3,salesVolume_lag4,salesVolume_lag5,salesVolume_lag6,salesVolume_lag7,salesVolume_lag8,salesVolume_lag9,salesVolume_lag10,salesVolume_lag11
0,0,compact,2020-01-01,100.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,compact,2020-02-01,103.271947,100.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,compact,2020-03-01,106.183698,103.271947,100.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,compact,2020-04-01,108.414710,106.183698,103.271947,100.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,compact,2020-05-01,109.719379,108.414710,106.183698,103.271947,100.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,1,suv,2023-08-01,159.808210,159.906074,158.913416,156.939515,154.201670,151.001280,147.690662,144.634271,142.168572,140.565004,140.000098,140.536042
188,1,suv,2023-09-01,158.630599,159.808210,159.906074,158.913416,156.939515,154.201670,151.001280,147.690662,144.634271,142.168572,140.565004,140.000098
189,1,suv,2023-10-01,156.502878,158.630599,159.808210,159.906074,158.913416,156.939515,154.201670,151.001280,147.690662,144.634271,142.168572,140.565004
190,1,suv,2023-11-01,153.659282,156.502878,158.630599,159.808210,159.906074,158.913416,156.939515,154.201670,151.001280,147.690662,144.634271,142.168572


In [7]:
grouped_sequence = data.groupby(["provinceId", "model"]).apply(
    lambda x: x.sort_values("Date")[
        ["salesVolume", "salesVolume_lag1", "salesVolume_lag2", "salesVolume_lag3"]
    ].to_numpy()
)

data_3d = np.stack(grouped_sequence.values)

data_3d

array([[[100.        ,          nan,          nan,          nan],
        [103.27194697, 100.        ,          nan,          nan],
        [106.18369803, 103.27194697, 100.        ,          nan],
        [108.41470985, 106.18369803, 103.27194697, 100.        ],
        [109.71937901, 108.41470985, 106.18369803, 103.27194697],
        [109.95407958, 109.71937901, 108.41470985, 106.18369803],
        [109.09297427, 109.95407958, 109.71937901, 108.41470985],
        [107.23085882, 109.09297427, 109.95407958, 109.71937901],
        [104.57272627, 107.23085882, 109.09297427, 109.95407958],
        [101.41120008, 104.57272627, 107.23085882, 109.09297427],
        [ 98.09432037, 101.41120008, 104.57272627, 107.23085882],
        [ 94.98722951,  98.09432037, 101.41120008, 104.57272627],
        [ 92.43197505,  94.98722951,  98.09432037, 101.41120008],
        [ 90.70985499,  92.43197505,  94.98722951,  98.09432037],
        [ 90.01045083,  90.70985499,  92.43197505,  94.98722951],
        [ 

In [8]:
from tensorflow.keras.utils import Sequence


class TimeDataset(Sequence):
    def __init__(self, data, train_sequence_length, predict_sequence_length, batch_size: int = 64):
        self.data = data
        self.train_seq_len = train_sequence_length
        self.pred_seq_len = predict_sequence_length
        self.batch_size = batch_size

        self.num_ids = data.shape[0]
        self.max_seq_len = data.shape[1]
        self.feature_dim = data.shape[2]

        self.samples_per_id = self.max_seq_len - self.train_seq_len - self.pred_seq_len + 1
        self.total_samples = self.num_ids * self.samples_per_id

        # Precompute all valid (id, start_idx) pairs
        self.indices = [(i, j) for i in range(self.num_ids) for j in range(self.samples_per_id)]

    def __getitem__(self, index):
        # batch-wise item
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]

        x_batch = []
        y_batch = []

        for id_idx, start_idx in batch_indices:
            x = self.data[id_idx, start_idx : start_idx + self.train_seq_len, 1:]
            y = self.data[
                id_idx, start_idx + self.train_seq_len : start_idx + self.train_seq_len + self.pred_seq_len, 0
            ]
            x_batch.append(x)
            y_batch.append(y)

        return np.nan_to_num(np.array(x_batch)), np.nan_to_num(np.array(y_batch))[..., None]

    def __len__(self):
        # depends on how many samples you want to extract from 1 ID
        return int(np.ceil(len(self.indices) / self.batch_size))

In [9]:
train_dataset = TimeDataset(data_3d, CFG.train_sequence_length, CFG.predict_sequence_length)
valid_dataset = TimeDataset(data_3d, CFG.train_sequence_length, CFG.predict_sequence_length)

print(train_dataset[0][0].shape)
print(train_dataset[0][1].shape)

(64, 12, 3)
(64, 3, 1)


In [10]:
def build_model():

    config = AutoConfig.for_model("rnn")
    config.rnn_type = "lstm"
    model = AutoModelForForecasting.from_config(config, prediction_length=CFG.predict_sequence_length)
    return model


model = build_model()
_ = model(train_dataset[0][0])
model.summary()

Model: "forecasting_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rnn (RNN)                       │ ?                      │        50,819 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,819 (198.51 KB)

 Trainable params: 50,819 (198.51 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
trainer = KerasTrainer(model)
history = trainer.train(train_dataset, valid_dataset, epochs=10)
trainer.save_model("./sales_model")

# Reconstruct the task model with its config and weights. It remains trainable.
restored_model = AutoModel.from_pretrained("./sales_model", sample_batch=train_dataset[0][0])

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 16386.3164 - mae: 126.4892

/Users/longxingtan/Repository/Time-series-prediction/.venv/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - loss: 16386.3164 - mae: 126.4892 - val_loss: 16374.5859 - val_mae: 126.4430
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - loss: 16374.5859 - mae: 126.4430 - val_loss: 16374.5859 - val_mae: 126.4430
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - loss: 16374.5850 - mae: 126.4430 - val_loss: 16374.5859 - val_mae: 126.4430
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - loss: 16374.5850 - mae: 126.4430 - val_loss: 16374.5859 - val_mae: 126.4430
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - loss: 16374.5850 - mae: 126.4430 - val_loss: 16374.5859 - val_mae: 126.4430
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - loss: 16374.5850 - mae: 126.4430 - val_loss: 16374.5859 - val_mae: 126.4430
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - loss: 16374.5850 - mae: 126.4430 - val_loss: 16374.5859 - val_mae: 126.4430
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - loss: 16374.5859 - mae: 126.4430 - val_loss: 16374.5859 - val_mae: 12

In [12]:
# Run inference with the restored model.
inference_x, _ = valid_dataset[0]
restored_predictions = restored_model(inference_x, training=False).numpy()
print("Restored prediction shape:", restored_predictions.shape)

Restored prediction shape: (64, 3, 1)
